In [ ]:
# Example 1: Public API surface — verify the canonical prompt constructors are exposed.
import questionary

expected_api = [
    'text', 'password', 'confirm', 'select', 'rawselect',
    'checkbox', 'path', 'autocomplete', 'press_any_key_to_continue',
    'prompt', 'unsafe_prompt', 'Choice', 'Separator', 'Style', 'Validator',
]
missing = [name for name in expected_api if not hasattr(questionary, name)]
assert not missing, f"Missing public API: {missing}"

for name in expected_api:
    obj = getattr(questionary, name)
    print(f"  {name}: {type(obj).__name__}")

assert callable(questionary.text)
assert callable(questionary.select)

In [ ]:
# Example 2: Choice and Separator objects — construct without prompting.
from questionary import Choice, Separator

c1 = Choice(title="Option A", value="a")
c2 = Choice(title="Option B", value="b", disabled="not available")
c3 = Choice(title="Default", value="d", checked=True)

assert c1.value == "a"
assert c2.value == "b"
assert c2.disabled == "not available"
assert c3.checked is True

sep = Separator()
sep_custom = Separator("--- section ---")

print(f"Choice 1: title={c1.title!r}, value={c1.value!r}")
print(f"Choice 2 disabled: {c2.disabled!r}")
print(f"Default separator title: {sep.title!r}")
print(f"Custom separator title: {sep_custom.title!r}")

In [ ]:
# Example 3: Question objects — construct without interaction.
import questionary
from questionary import Question

q_text = questionary.text("What is your name?", default="Anonymous")
q_confirm = questionary.confirm("Do you agree?", default=True)
q_select = questionary.select(
    "Pick one:",
    choices=["apple", "banana", "cherry"],
    default="banana",
)
q_checkbox = questionary.checkbox(
    "Pick many:",
    choices=["red", "green", "blue"],
)

assert isinstance(q_text, Question)
assert isinstance(q_confirm, Question)
assert isinstance(q_select, Question)
assert isinstance(q_checkbox, Question)

for label, q in [("text", q_text), ("confirm", q_confirm), ("select", q_select), ("checkbox", q_checkbox)]:
    print(f"  {label} -> {type(q).__name__}, application={type(q.application).__name__}")

In [ ]:
# Example 4: Custom Validator — exercise the validator-callable pattern.
from prompt_toolkit.document import Document
from prompt_toolkit.validation import ValidationError, Validator

import questionary

class LengthValidator(Validator):
    def validate(self, document: Document):
        if len(document.text) < 3:
            raise ValidationError(message="Too short", cursor_position=len(document.text))

validator = LengthValidator()
validator.validate(Document(text="hello"))
print("  'hello' passed validation")

raised = False
try:
    validator.validate(Document(text="hi"))
except ValidationError as e:
    raised = True
    print(f"  'hi' rejected: {e.message}")

assert raised, "Validator should have raised on short input"

q = questionary.text("Your name:", validate=validator)
assert q is not None
print(f"  Built text question with validator: {type(q).__name__}")

In [ ]:
# Example 5: Style customization, a form, and a prompt answered from piped input.
from concurrent.futures import ThreadPoolExecutor

from prompt_toolkit.input import create_pipe_input
from prompt_toolkit.output import DummyOutput
from prompt_toolkit.styles import Style

import questionary

custom_style = Style([
    ("qmark", "fg:#673ab7 bold"),
    ("question", "bold"),
    ("answer", "fg:#f44336 bold"),
    ("pointer", "fg:#673ab7 bold"),
    ("highlighted", "fg:#673ab7 bold"),
    ("selected", "fg:#cc5454"),
    ("separator", "fg:#cc5454"),
    ("instruction", ""),
    ("text", ""),
    ("disabled", "fg:#858585 italic"),
])

assert custom_style is not None

q = questionary.select(
    "Choose your weapon:",
    choices=["sword", "bow", "staff"],
    style=custom_style,
)
assert q is not None

# questionary.form groups named questions; nothing is asked until .ask() runs.
form = questionary.form(username=questionary.text("Username?"), agree=questionary.confirm("OK?"))
assert isinstance(form, questionary.Form), type(form)
assert [field.key for field in form.form_fields] == ["username", "agree"], form.form_fields

# unsafe_prompt asks a list of question dicts; here the keystrokes come from a pipe instead of a terminal.
questions_spec = [
    {"type": "text", "name": "username", "message": "Username?"},
    {"type": "confirm", "name": "agree", "message": "OK?", "default": False},
]


def answer(keys):
    with create_pipe_input() as pipe:
        pipe.send_text(keys)
        return questionary.unsafe_prompt(questions_spec, input=pipe, output=DummyOutput())


# Jupyter already runs an event loop in this thread, so the prompt runs on a worker thread.
with ThreadPoolExecutor(max_workers=1) as pool:
    answers = pool.submit(answer, "alice\ry").result(timeout=30)
print(f"  Answers from piped input: {answers}")
assert answers == {"username": "alice", "agree": True}, answers
print(f"  Custom style attached; the prompt answered {len(answers)} questions")
print("  questionary smoke test PASSED")